# 10 — draft persistence: an edit a person made is never lost

The safety half of `docs/architecture/editing-performance.md`, and the half
that matters. Every check here reads the value back rather than a flag: a
`saveState` of `saved` is not evidence, the string in `localStorage` or the
string the agent answers is.

Everything here must be green. The two checks that were red in the baseline of
2026-09-08 were acceptance of changes that have since been built, the catalogue
leaving the persisted payload and every field write waiting out one quiet
window, and both are green now.

**What changed under this notebook, 2026-09-08.** The shared draft's backend is
no longer the site's repository. A save writes the site's Durable Object and
bumps a revision, a publish writes the one migration and the one commit, and the
object pushes revisions to the other editors over a WebSocket with the poll left
as the fallback. So `headSha` and `blobSha` are gone from every check here and
the revision took their place, and three scenarios the design names are new: the
object pushing, the socket refused, the object restarted between two saves, and
two editors racing on one field.

**The fixture changed with it.** These scenarios used to read
`booklimo.at/jaen-data/live.json` and `live-media.json` off the site. The
transition deleted both, because a head named in `patches.txt` made every
unfinished edit part of the published site, and kept them beside the site at
`booklimo.at-transition-before.head/`. The first cell merges those two into one
draft, which is the shape the object's own `snapshot(site)` answers, and every
node scenario reads that.

**What the node half proves, and what it does not.** `editing-shim.ts` holds a
stand in for the site's draft object: a monotonic revision, an answer that is
either a delta or a full replacement, a write against a stale base folded rather
than refused, and a socket that can be told to refuse. It is not the agent, and
no check written against it says anything about the agent's own code. What it
proves is what the **client** does when the object behaves in each of the ways
the design says it can, including the two it is not supposed to.

## How each scenario is driven, and why

A scenario that can be driven in node is driven in node, because a node check
runs anywhere: no browser, no build, no credentials, no writes on a live site.
`support/editing-harness.ts` bundles `packages/jaen/src/redux` itself, so the
store, the persister, the recorder and the flusher are the real ones; only the
browser globals and the agent's HTTP are replaced, and the replaced `fetch` is
what makes "offline" mean what it means to the client, a rejected call rather
than an error answer.

One scenario per process. The store is a module singleton, so a second
scenario in the same process would inherit the first one's state. A reload is
two processes over one storage file, which is exactly what a reload is.

What cannot be driven in node is driven in a browser against the live agent:
a real reload, a real hidden tab, a real offline context and a second editor.
Those run on a local production build of booklimo.at signed in as the booklimo
human admin, they make commits on booklimo through the live agent, and they set
the field back and read it back out of an emptied browser at the end. Nothing
is written on limosen.

In [1]:
import json, os, pathlib

import jaen_testkit as k

k.start_run('10-draft-persistence')

REPO = pathlib.Path(k.CONFIG['repo_root'])
SITE = pathlib.Path(os.environ.get('JAEN_SITE_BUILD', '/home/snekmin/git/limosen-v3/booklimo.at'))
WORK = pathlib.Path(os.environ.get('JAEN_WORK_DIR', '/tmp/jaen-editing'))
WORK.mkdir(parents=True, exist_ok=True)
BUNDLE = WORK / 'editing-harness.cjs'
PLAYWRIGHT_PYTHON = os.environ.get(
    'JAEN_PLAYWRIGHT_PYTHON', '/home/snekmin/git/taxi-app/tests/.venv/bin/python')

with k.section('the harness'):
    with k.check('the harness bundles out of the jaen source') as c:
        esbuild = REPO / 'node_modules' / '.bin' / 'esbuild'
        if not esbuild.is_file():
            c.skip('no esbuild in the checkout')
        r = c.require(k.sh(
            '%s tests/support/editing-harness.ts --bundle --platform=node '
            '--format=cjs --target=node20 --outfile=%s --log-level=warning'
            % (esbuild, BUNDLE), cwd=str(REPO), timeout=180))
        c.expect_true(BUNDLE.is_file(), 'bundled %d B' % BUNDLE.stat().st_size)


def harness(scenario, env=None, timeout=120):
    # One scenario, one process.
    base = {'JAEN_HARNESS_DRAFT': str(DRAFT)}
    base.update(env or {})
    return k.sh('node %s %s' % (BUNDLE, scenario), cwd=str(REPO), env=base,
                timeout=timeout, label='harness %s' % scenario)


def scenario(name, env=None, timeout=120):
    r = harness(name, env, timeout)
    return (json.loads(r.text) if r.ok and r.text.startswith('{') else None), r

# The fixture: one draft, out of the two files the transition kept
#
# `support/editing-harness.ts` used to read `booklimo.at/jaen-data/live.json`
# and `live-media.json` straight off the site. The transition of
# `docs/architecture/draft-state.md` deleted both, because a head named in
# `patches.txt` made every unfinished edit part of the published site, and kept
# them outside every repository at `booklimo.at-transition-before.head/`
# together with their authors map.
#
# So the fixture is built here and passed as one file, `{pages, site, widgets}`,
# which is the shape the draft object's own `snapshot(site)` answers. It is
# deliberately the *draft* and not the site's whole replayed data: what the CMS
# store holds is the unpublished draft, and the sourced jaen data of a build,
# which the transition also kept beside the site, is three and a half times
# larger and would inflate every number here against the baseline.
HEAD = pathlib.Path(os.environ.get(
    'JAEN_HARNESS_HEAD',
    '/home/snekmin/git/limosen-v3/booklimo.at-transition-before.head'))
DRAFT = WORK / 'booklimo-draft.json'


def build_fixture():
    """The two head files merged into one draft, pages by id."""
    live = json.loads((HEAD / 'live.json').read_text())
    media = json.loads((HEAD / 'live-media.json').read_text())

    pages = {}
    for page in (live['data'].get('pages') or []) + (media['data'].get('pages') or []):
        at = pages.setdefault(page['id'], {'id': page['id']})
        for key, value in page.items():
            if key == 'jaenFields':
                at.setdefault('jaenFields', {}).update(value or {})
            else:
                at[key] = value

    draft = {'pages': list(pages.values()),
             'site': live['data'].get('site') or {'siteMetadata': {}},
             'widgets': live['data'].get('widgets') or []}
    DRAFT.write_text(json.dumps(draft))
    return draft


with k.section('the fixture'):
    with k.check('the head the transition kept is beside the site') as c:
        c.expect_true((HEAD / 'live.json').is_file(), 'live.json at %s' % HEAD)
        c.expect_true((HEAD / 'live-media.json').is_file(), 'live-media.json at %s' % HEAD)
        if not (HEAD / 'live.json').is_file():
            c.fail('no fixture, every node scenario below will fail', abort=True)
        build_fixture()
        c.note('%d B of draft at %s' % (DRAFT.stat().st_size, DRAFT))

# Whether the built site still carries the agent option
#
# The transition of `docs/architecture/draft-state.md` removed the `agent`
# option from both sites' `gatsby-config.ts` and rebuilt them, which is that
# design's own rollback: without it `__JAEN_AGENT__` is undefined, the CMS keeps
# its draft in `localStorage` alone, and no request goes to any agent host. So
# every check below that needs a live shared draft has nothing to talk to, and
# it says that rather than failing or pretending.
def site_has_agent():
    public = SITE / 'public'
    if not public.is_dir():
        return False
    for path in public.rglob('*.js'):
        try:
            if 'jaen-agent' in path.read_text(errors='ignore'):
                return True
        except OSError:
            continue
    return False


AGENT_IN_BUILD = site_has_agent()


## The client and the agent agree on the wire

The client is hand written on purpose, so a field renamed on the agent's side is
not a compile error anywhere. It is a `GRAPHQL_VALIDATION_FAILED` at runtime
that refuses the **whole** operation rather than one field, which for `save`
means an editor's work is simply never sent. This asks the agent's own generated
schema rather than a person.

Two of Pylon's rules are the ones that bite and both are asserted here by
consequence. A `number` argument becomes the scalar `Number` and not `Int`, so
an operation declaring `Int` is refused before the resolver is reached. And a
`Record<string, X>` becomes `JSONObject` and takes no subselection while a typed
interface becomes an object type and demands one, so guessing wrong about one
field of the draft delta fails the read entirely.

It SKIPs where the agent has not been built in this checkout, because
`.pylon/schema.graphql` is a build artifact.


In [2]:
SCHEMA = None

with k.section('the wire'):
    with k.check("every document the client sends validates against the agent's schema") as c:
        r = c.require(k.sh('node tests/support/validate-agent-documents.cjs',
                           cwd=str(REPO), timeout=120, label='validate documents'),
                      'the validator')
        SCHEMA = json.loads(r.text)
        if SCHEMA.get('skipped'):
            c.skip(SCHEMA['skipped'])
        names = [d['name'] for d in SCHEMA['documents']]
        c.note('%d document(s): %s' % (len(names), ', '.join(names)))
        for document in SCHEMA['documents']:
            c.expect_equal(document['errors'], [], document['name'])
        c.expect_true(len(names) >= 5,
                      'draft, save, viewer, subscribe and publish')

print(json.dumps(SCHEMA, indent=1))

{
 "schema": "/home/snekmin/git/limosen-v3/jaen/packages/jaen-agent/.pylon/schema.graphql",
 "client": "/home/snekmin/git/limosen-v3/jaen/packages/jaen/src/clients/agent/index.ts",
 "documents": [
  {
   "name": "JaenAgentDraft",
   "errors": []
  },
  {
   "name": "JaenAgentDiscardPreview",
   "errors": []
  },
  {
   "name": "JaenAgentDiscard",
   "errors": []
  },
  {
   "name": "JaenAgentSave",
   "errors": []
  },
  {
   "name": "JaenAgentViewer",
   "errors": []
  },
  {
   "name": "JaenAgentSubscribe",
   "errors": []
  },
  {
   "name": "JaenAgentPublish",
   "errors": []
  }
 ],
 "invalid": 0
}


## What the persisted payload carries

The plan wants three things of it: the outbox, so a change that has not
reached the agent survives a reload; the local edits, for the same reason; and
**not** the media catalogue, which is 99% of it, is never edited by hand and
comes back on the next poll.

The scenario writes a field with the agent unreachable, so the change is still
in the outbox when the payload is read. The catalogue check is the acceptance
of change 2 and is red until that change is made.

In [3]:
PAYLOAD, _r = scenario('payload')

with k.section('the persisted payload'):
    with k.check('the payload carries the unsent change and the edit') as c:
        if PAYLOAD is None:
            c.fail('the harness did not answer', abort=True)
        c.expect_equal(PAYLOAD['localEdit'], 'the payload check',
                       'the edit is in the payload, read back out of it')
        c.expect_equal(PAYLOAD['outboxLength'], 1, 'changes waiting in the outbox')
        c.expect_equal(PAYLOAD['outboxKinds'], ['fieldWrite'], 'what is waiting')

    with k.check('the payload does not carry the media catalogue') as c:
        if PAYLOAD is None:
            c.skip('the harness did not answer')
        c.note('%d B payload, %d media nodes in it, %d B of catalogue'
               % (PAYLOAD['bytes'], PAYLOAD['mediaNodeCount'], PAYLOAD['catalogueBytes']))
        c.expect_equal(PAYLOAD['mediaNodeCount'], 0,
                       'change 2 of the plan: the catalogue is not persisted')

print(json.dumps(PAYLOAD, indent=1))

{
 "scenario": "payload",
 "bytes": 1086,
 "hasLocalEdit": true,
 "localEdit": "the payload check",
 "outboxLength": 1,
 "outboxKinds": [
  "fieldWrite"
 ],
 "mediaNodeCount": 0,
 "catalogueBytes": 2,
 "keys": [
  "site",
  "page",
  "status",
  "popup",
  "widget",
  "remote"
 ]
}


## An edit followed by a reload

Two processes over one storage file. The first writes a field and ends; the
second builds the store from what is on disk, the way `loadState` does on a
fresh page, and is asked what the field says.

In [4]:
STORE_FILE = WORK / 'reload-store.json'
STORE_FILE.unlink(missing_ok=True)

with k.section('a reload'):
    with k.check('an edit survives the browser being started again') as c:
        first, r1 = scenario('editThenExit', {'JAEN_HARNESS_STORE': str(STORE_FILE),
                                              'JAEN_HARNESS_VALUE': 'survives a reload'})
        if first is None:
            c.fail('the first process did not answer: %s' % r1.evidence(), abort=True)
        second, r2 = scenario('readBack', {'JAEN_HARNESS_STORE': str(STORE_FILE)})
        if second is None:
            c.fail('the second process did not answer: %s' % r2.evidence(), abort=True)
        c.expect_equal(second['value'], 'survives a reload',
                       'the value the second process read out of storage')
        c.note('and %d media nodes came back with it' % second['mediaNodeCount'])

## An edit followed by the tab going away

The dangerous one. Today the write is synchronous inside the dispatch, so the
value is in storage in the same frame. After change 1 it will be written in an
idle callback with a 250 ms timeout, and `visibilitychange` to hidden and
`pagehide` will write at once. The check is written so that it holds for both:
the value must be in storage once the tab is hidden with no waiting at all, and
in any case within the idle deadline the plan gives itself.

In [5]:
HIDDEN, _r = scenario('hiddenTab')

with k.section('a hidden tab'):
    with k.check('a hidden tab has the edit in storage with no waiting') as c:
        if HIDDEN is None:
            c.fail('the harness did not answer', abort=True)
        c.note('%d listener(s) heard visibilitychange' % HIDDEN['listeners'])
        c.expect_equal(HIDDEN['afterHidden'], 'typed and then the tab went away',
                       'read back out of storage in the same frame as the event')

    with k.check('the edit is in storage within the idle deadline anyway') as c:
        if HIDDEN is None:
            c.skip('the harness did not answer')
        c.expect_equal(HIDDEN['afterIdleDeadline'], 'typed and then the tab went away',
                       'read back 300 ms after the dispatch')
        if HIDDEN['immediately'] is None:
            c.warn('the write is asynchronous: nothing was in storage in the '
                   'dispatch\'s own frame, which is what change 1 makes true')
        else:
            c.note('the write is synchronous today: it was there at once')

## An edit made offline

The outbox is the offline queue, and it is the offline queue because
`persist-state` writes it. The scenario writes a field with `fetch` rejecting,
reads the value and the queue back out of storage, then lets the agent come
back.

Two ways back are measured. Without any help the client retries on its own
backoff, and the first retry is measured here rather than assumed: the plan and
`draft-state.md` both say the first retry is at two seconds, and the code
indexes `RETRY_SECONDS` with a failure count it has already incremented, so it
is at five. That is written down as an observation, not fixed here.

In [6]:
OFFLINE, _r = scenario('offlineDrain', timeout=180)
ONLINE, _r2 = scenario('onlineEvent', timeout=120)

with k.section('offline'):
    with k.check('an edit made offline is kept, in the store and in storage') as c:
        if OFFLINE is None:
            c.fail('the harness did not answer', abort=True)
        while_offline = OFFLINE['whileOffline']
        c.expect_equal(while_offline['persistedValue'], 'written while offline',
                       'read back out of storage while the agent is unreachable')
        c.expect_equal(while_offline['outboxLength'], 1, 'waiting in the outbox')
        c.expect_equal(while_offline['saveState'], 'offline', 'what the toolbar says')

    with k.check('the queue drains on its own when the agent comes back') as c:
        if OFFLINE is None:
            c.skip('the harness did not answer')
        after = OFFLINE['afterReturn']
        c.expect_equal(after['outboxLength'], 0, 'the outbox is empty again')
        c.expect_equal(after['persistedValue'], 'written while offline',
                       'and the value is still the one that was typed')
        c.note('drained %.1f s after the network came back' % (OFFLINE['drainedAfterMs'] / 1000))

    with k.check('the documented first retry is the one the code takes') as c:
        if OFFLINE is None:
            c.skip('the harness did not answer')
        seconds = OFFLINE['drainedAfterMs'] / 1000
        if 1.5 <= seconds <= 3.5:
            c.ok('first retry at %.1f s, as documented' % seconds)
        else:
            c.warn('first retry at %.1f s, while draft-state.md and the plan both '
                   'say 2 s: RETRY_SECONDS is indexed with an already incremented '
                   'failure count. Nothing is lost by it, the queue only waits '
                   'longer than the document claims.' % seconds)

    with k.check('the browser saying online drains it at once') as c:
        if ONLINE is None:
            c.fail('the harness did not answer', abort=True)
        c.expect_equal(ONLINE['outboxLength'], 0, 'the outbox is empty')
        c.expect_equal(ONLINE['persistedValue'], 'written while offline',
                       'the value read back out of storage')
        c.note('drained %d ms after the online event' % ONLINE['drainedAfterMs'])

## A save inside the quiet window is one write of the draft

Change 3 of `editing-performance.md`: the quiet window applies to every field
write and not only to a streaming one, so a person moving through three fields
makes one call instead of three. The scenario writes three different fields
120 ms apart, well inside the window the code ships with, and counts the calls
the agent received.

It says one call and no longer one commit, because a save is no longer a
commit: it writes the site's draft object and the repository is written by a
publish alone. What the window saves is a round trip and a whole-store write
per field, which is what it always saved; that it also saved a commit was the
wrong direction and is gone.


In [7]:
QUIET, _r = scenario('quietWindow', timeout=180)

with k.section('the quiet window'):
    with k.check('three field writes inside the window are one commit') as c:
        if QUIET is None:
            c.fail('the harness did not answer', abort=True)
        c.note('%d call(s) to the agent, %s change(s) each'
               % (QUIET['saveCalls'], QUIET['changesPerCall']))
        c.expect_equal(QUIET['saveCalls'], 1,
                       'change 3 of editing-performance.md: one call, not three')

    with k.check('none of the three values is lost, whatever the batching') as c:
        if QUIET is None:
            c.skip('the harness did not answer')
        c.expect_equal(QUIET['values'],
                       {'FleetTitle': 'one', 'ServicesTitle': 'two', 'AboutTitle': 'three'},
                       'all three read back out of the store')
        c.expect_equal(QUIET['outboxLength'], 0, 'and all three left the outbox')

## A poll arriving mid-edit

The poller hydrates the remote draft into the same store the editor is typing
into. `hydrate` folds the outbox back on top of the remote answer, so a change
that has not reached the agent yet cannot be overwritten by an answer that does
not contain it. The scenario keeps the agent unreachable so the change stays
unsent, then hydrates the draft as the poller would and asks the store and the
storage what the field says.

In [8]:
POLL, _r = scenario('pollMidEdit')

with k.section('a poll mid-edit'):
    with k.check('a poll does not overwrite an unsent change') as c:
        if POLL is None:
            c.fail('the harness did not answer', abort=True)
        c.note('the remote draft says %r' % POLL['remoteValueBeforeMerge'])
        c.expect_equal(POLL['value'], 'typed and not yet sent',
                       'the store after the hydration')
        c.expect_equal(POLL['persistedValue'], 'typed and not yet sent',
                       'and storage, read back')
        c.expect_equal(POLL['outboxLength'], 1, 'the change is still waiting')

## Discard clears the outbox and nothing else

`RESET_STATE` keeps `active`, the head and the authors and drops everything a
browser had not published. With the agent configured the toolbar no longer
offers it, but the action is still there and the escape below relies on it.

In [9]:
DISCARD, _r = scenario('discard')

with k.section('discard'):
    with k.check('discard drops the unsent changes and keeps the revision') as c:
        if DISCARD is None:
            c.fail('the harness did not answer', abort=True)
        c.expect_equal(DISCARD['before']['outboxLength'], 1, 'something to discard')
        c.expect_equal(DISCARD['after']['outboxLength'], 0, 'the outbox is empty')
        c.expect_equal(DISCARD['after'].get('value'), None,
                       'the edit is gone from the store, read back')
        c.expect_equal(DISCARD['after'].get('persistedValue'), None,
                       'and out of storage')
        c.expect_equal(DISCARD['after']['revision'], 7,
                       'the revision the read gave us survives, so the next read asks for a delta')
        c.expect_equal(DISCARD['after']['authors'], 1, 'and so do the authors')

## The escape: no agent at all

The plan's rollback. A site built without the `agent` plugin option behaves
exactly as it did before the shared draft: no recorder, no flusher, nothing on
the network, and `localStorage` as the only store. The scenario runs the same
harness with the define removed.

In [10]:
NOAGENT, _r = scenario('noAgent', {'JAEN_HARNESS_AGENT': '0'})

with k.section('the escape'):
    with k.check('without the agent the CMS still saves to localStorage alone') as c:
        if NOAGENT is None:
            c.fail('the harness did not answer', abort=True)
        c.expect_true(not NOAGENT['agentConfigured'], 'no agent is configured')
        c.expect_equal(NOAGENT['persistedValue'], 'saved without the agent',
                       'the edit read back out of storage')
        c.expect_equal(NOAGENT['saveCalls'], 0, 'and nothing left the browser')
        c.expect_equal(NOAGENT['outboxLength'], 0, 'no outbox is kept')

    with k.check('and the catalogue is still persisted, because nothing else holds it') as c:
        if NOAGENT is None:
            c.skip('the harness did not answer')
        # The safety condition change 2 needs and the plan leaves implicit.
        # With the agent, the catalogue is dropped from the payload and the
        # poller hands it back. Without it, localStorage is the only store
        # there is, so dropping it would drop every picture the browser holds
        # and has not published.
        c.note('%d B payload, %d media nodes in it'
               % (NOAGENT['bytes'], NOAGENT['mediaNodeCount']))
        c.expect_true(NOAGENT['mediaNodeCount'] > 0,
                      'the escape keeps the media library')

## The object is pushing, and the frame is a number

The path the poll is the fallback for. The object pushes a revision over its
socket and never content, which is `subscribe(site)` of the design's four
operation interface, and what the client does with a frame is ask the same
`draft` query the poll asks. So this measures the whole round: a number goes
out, and the other editor's field is on this screen.

Two things beside the timing, and both are about the credential. The socket is
opened with a **ticket** minted by an ordinary GraphQL call carrying the
bearer, not with the person's token, and the ticket travels as a subprotocol
and never as a query parameter, for the reason `private-storage.md` gives about
a credential in a URL.


In [11]:
CARRIES, _r = scenario('socketCarries', timeout=120)

with k.section('the socket'):
    with k.check('the socket comes up and carries the other editor') as c:
        if CARRIES is None:
            c.fail('the harness did not answer', abort=True)
        c.note('connection %s, %d socket(s), %d ticket(s), %s'
               % (CARRIES['up']['connection'], CARRIES['up']['socketsOpened'],
                  CARRIES['up']['tickets'], CARRIES['up']['url']))
        c.expect_equal(CARRIES['up']['connection'], 'socket', 'the client says the object is pushing')
        c.expect_equal(CARRIES['pushedTo'], 1, 'the frame reached one open socket')

    with k.check("the other editor's field is read back off this store") as c:
        if CARRIES is None:
            c.skip('the harness did not answer')
        c.note('%.1f ms from the frame to the value, %d draft call(s) after it'
               % (CARRIES['arrivedAfterMs'], CARRIES['draftCallsAfterFrame']))
        c.expect_equal(CARRIES['value'], 'pushed over the socket',
                       'the value, not a flag')
        c.expect_true(0 <= CARRIES['arrivedAfterMs'] < 2000,
                      'under the two seconds draft-state.md asks of two editors')
        c.expect_equal(CARRIES['draftCallsAfterFrame'], 1,
                       'one read per frame: the frame carries a number and the read carries the data')

    with k.check('the token never travels on the socket') as c:
        if CARRIES is None:
            c.skip('the harness did not answer')
        protocols = CARRIES['up']['protocols']
        c.note('subprotocols %s' % protocols)
        c.expect_true(any(p.startswith('ticket.') for p in protocols),
                      'the ticket is a subprotocol')
        c.expect_true(not any(p.startswith('bearer.') for p in protocols),
                      'and it is not the person\'s token')
        c.expect_true('?' not in (CARRIES['up']['url'] or ''),
                      'nothing in the query string: %s' % CARRIES['up']['url'])

print(json.dumps(CARRIES, indent=1))

{
 "scenario": "socketCarries",
 "up": {
  "connection": "socket",
  "socketsOpened": 1,
  "protocols": [
   "jaen-draft.v1",
   "ticket.ticket-1"
  ],
  "url": "wss://agent.example.invalid/draft/booklimo.at",
  "tickets": 1
 },
 "pushedTo": 1,
 "arrivedAfterMs": 9.777599999999993,
 "value": "pushed over the socket",
 "revision": 1,
 "draftCallsAfterFrame": 1
}


## The socket is refused, so the poll carries it

`draft-state.md` keeps the poll as the fallback for a browser whose socket is
refused, and this is that browser: the agent answers `subscribe` with a
validation error, which is exactly what an agent that does not have the verb
yet answers, and this CMS is then a poll and nothing else.

What is asserted is not that a flag reads `poll`. It is that the other editor's
field arrives anyway, read back off this browser's own store and out of its
`localStorage`, and that the authors map came with it.


In [12]:
REFUSED, _r = scenario('socketRefused', timeout=120)

with k.section('the poll as the fallback'):
    with k.check('a refused socket leaves the client on the poll') as c:
        if REFUSED is None:
            c.fail('the harness did not answer', abort=True)
        c.note('%d subscribe call(s), %d socket(s) opened'
               % (REFUSED['settled']['subscribeCalls'], REFUSED['settled']['socketsOpened']))
        c.expect_equal(REFUSED['settled']['connection'], 'poll', 'the client says so')
        c.expect_equal(REFUSED['settled']['socketsOpened'], 0,
                       'no socket was opened at all, the ticket was refused first')

    with k.check("the other editor's field arrives on the poll alone") as c:
        if REFUSED is None:
            c.skip('the harness did not answer')
        c.note('%.0f ms, %d draft call(s)' % (REFUSED['arrivedAfterMs'], REFUSED['draftCalls']))
        c.expect_equal(REFUSED['value'], 'written by the other editor',
                       'read back off the store')
        c.expect_equal(REFUSED['persistedValue'], 'written by the other editor',
                       'and out of localStorage')
        c.expect_true(REFUSED['arrivedAfterMs'] >= 0, 'it arrived')

    with k.check('who wrote it came with it') as c:
        if REFUSED is None:
            c.skip('the harness did not answer')
        c.expect_true('JaenPage //IMA:TextField/FleetTitle' in REFUSED['authors'],
                      'the authors map names the field: %s' % REFUSED['authors'])

print(json.dumps(REFUSED, indent=1))

{
 "scenario": "socketRefused",
 "settled": {
  "connection": "poll",
  "subscribeCalls": 1,
  "socketsOpened": 0
 },
 "arrivedAfterMs": 205.33209999999997,
 "value": "written by the other editor",
 "persistedValue": "written by the other editor",
 "connection": "poll",
 "revision": 1,
 "authors": [
  "JaenPage //IMA:TextField/FleetTitle"
 ],
 "draftCalls": 4
}


## The object restarted between two saves

The design's worst case for the draft store, and the reason it writes a
snapshot outside itself. This asks the narrower question, which is the
client's to answer.

A revision is monotonic while its object lives, so a read that comes back
**below** the revision this browser already had is not a race, it is a
different object answering in the place of one that is gone. Its answer is
older than what is on this screen, so applying it would take an edit away from
the person who made it. The client skips exactly that answer, adopts the
object's revision so the next save carries a base the new object recognises,
and says so.

What this cannot recover, and says so out loud, is what the *other* editors had
written into the object that died. No browser holds that. The snapshot is the
design's answer to it and it is the agent's to build.


In [13]:
RESTART, _r = scenario('objectRestart', timeout=180)

with k.section('the object restarted'):
    with k.check('the first save was taken') as c:
        if RESTART is None:
            c.fail('the harness did not answer', abort=True)
        c.expect_equal(RESTART['afterFirstSave']['value'], 'before the object went away',
                       'the value this browser wrote')
        c.expect_true(RESTART['afterFirstSave']['revision'] >= 1,
                      'the object answered a revision: %s' % RESTART['afterFirstSave']['revision'])

    with k.check('the client noticed the revision going backwards') as c:
        if RESTART is None:
            c.skip('the harness did not answer')
        c.note('objectRestartedAt %s, revision now %s'
               % (RESTART['afterRestart']['objectRestartedAt'], RESTART['afterRestart']['revision']))
        c.expect_true(RESTART['afterRestart']['objectRestartedAt'] is not None,
                      'a read below the held revision was recognised')
        c.expect_true(RESTART['afterRestart']['revision'] < RESTART['afterFirstSave']['revision'],
                      "the object's own revision was adopted")

    with k.check('the edit the person made is still there') as c:
        if RESTART is None:
            c.skip('the harness did not answer')
        c.expect_equal(RESTART['afterRestart']['value'], 'before the object went away',
                       'on the screen, read back off the store')
        c.expect_equal(RESTART['afterRestart']['persistedValue'], 'before the object went away',
                       'and in localStorage')

    with k.check('and the next save goes through to the new object') as c:
        if RESTART is None:
            c.skip('the harness did not answer')
        after = RESTART['afterSecondSave']
        c.note('%d save(s) reached the object, it is at revision %s'
               % (after['objectSaves'], after['objectRevision']))
        c.expect_equal(after['value'], 'after the object came back', 'the second value')
        c.expect_equal(after['persistedValue'], 'after the object came back', 'in localStorage')
        c.expect_equal(after['outboxLength'], 0, 'nothing left waiting')
        c.expect_equal(after['saveState'], 'saved', 'and the toolbar says so')
        c.expect_equal(after['objectSaves'], 2, 'both saves reached an object')

print(json.dumps(RESTART, indent=1))

{
 "scenario": "objectRestart",
 "afterFirstSave": {
  "revision": 1,
  "value": "before the object went away",
  "objectRevision": 1
 },
 "afterRestart": {
  "objectRestartedAt": "2026-09-08T19:57:52.731Z",
  "revision": 0,
  "value": "before the object went away",
  "persistedValue": "before the object went away"
 },
 "afterSecondSave": {
  "revision": 1,
  "value": "after the object came back",
  "persistedValue": "after the object came back",
  "outboxLength": 0,
  "saveState": "saved",
  "objectSaves": 2,
  "objectRevision": 1
 }
}


## Two editors racing on one field

The race is inside the quiet window, which is where a race actually happens:
this browser types, and before its batch is sent the other editor's write
reaches the object. The socket is refused and the poll is set slow on purpose,
because a poll faster than the window would read the other editor's value first
and there would be no race to measure.

The object folds a stale base on rather than refusing it, so last write wins,
and it answers `overwrote` so the CMS can say what the race cost. Three things
are asked and none of them is a flag: this browser's value is the one that
stands, the field it took is named, and a read follows the rebase **at once**
rather than at the next interval, because a browser that has been rebased holds
a copy that is behind by definition.


In [14]:
RACE, _r = scenario('twoEditorsRace', {'JAEN_HARNESS_POLL_MS': '5000'}, timeout=180)

with k.section('two editors on one field'):
    with k.check('this browser saved against a base the object had passed') as c:
        if RACE is None:
            c.fail('the harness did not answer', abort=True)
        c.note('sent baseRevision %s, the object was at %s'
               % (RACE['baseSent'], RACE['objectRevision']))
        c.expect_equal(RACE['baseSent'], RACE['baseHeld'],
                       'the save carried the revision this browser held')
        c.expect_true(RACE['objectSaves'] == 1, 'exactly one save reached the object')

    with k.check('the rebase named the field it took') as c:
        if RACE is None:
            c.skip('the harness did not answer')
        c.note('overwrote %s' % RACE['rightAfterSave']['overwrote'])
        c.expect_true('JaenPage //IMA:TextField/FleetTitle' in RACE['rightAfterSave']['overwrote'],
                      'the CMS can say what the race cost')

    with k.check('a read follows the rebase at once') as c:
        if RACE is None:
            c.skip('the harness did not answer')
        c.note('%d draft call(s) between the save and the settle'
               % RACE['draftCallsAfterSave'])
        c.expect_true(RACE['draftCallsAfterSave'] >= 1,
                      'the browser asked for the delta rather than waiting out an interval')

    with k.check('nothing was lost and last write stands') as c:
        if RACE is None:
            c.skip('the harness did not answer')
        c.expect_equal(RACE['settled']['value'], 'and this browser was second',
                       'read back off the store')
        c.expect_equal(RACE['settled']['persistedValue'], 'and this browser was second',
                       'and out of localStorage')
        c.expect_equal(RACE['settled']['outboxLength'], 0, 'nothing left waiting')
        c.expect_equal(RACE['settled']['saveState'], 'saved', 'and the toolbar says so')

print(json.dumps(RACE, indent=1))

{
 "scenario": "twoEditorsRace",
 "baseSent": 0,
 "baseHeld": 0,
 "rightAfterSave": {
  "value": "and this browser was second",
  "overwrote": [
   "JaenPage //IMA:TextField/FleetTitle"
  ],
  "revision": 2
 },
 "draftCallsAfterSave": 1,
 "settled": {
  "value": "and this browser was second",
  "persistedValue": "and this browser was second",
  "revision": 2,
  "outboxLength": 0,
  "saveState": "saved"
 },
 "objectRevision": 2,
 "objectSaves": 1
}


## The four the browser has to answer

A real reload, a real hidden tab, a real offline context and a second editor,
on a local production build of booklimo.at against the live agent, signed in as
the booklimo human admin. The run makes commits on booklimo, sets the field
back to the value it found and proves it by loading the site again in a browser
whose storage was emptied, so the value it reports as read back is the
repository's answer.

It SKIPs cleanly when playwright, the build or the human admin's credentials
are missing, the way the rest of the suite skips.

In [15]:
SAFETY = None

with k.section('the browser'):
    with k.check('a reload, a hidden tab, an offline edit and a second editor') as c:
        if not os.path.isfile(PLAYWRIGHT_PYTHON):
            c.skip('no playwright interpreter at %s' % PLAYWRIGHT_PYTHON)
        if not (SITE / 'public' / 'index.html').is_file():
            c.skip('no production build of booklimo.at in %s' % (SITE / 'public'))
        if not os.path.isfile(os.path.expanduser('~/.config/taxi-app/humans.env')):
            c.skip('no booklimo human admin configured')
        if not AGENT_IN_BUILD:
            c.skip('this build carries no agent option: the transition removed it '
                   'from both sites, so there is no shared draft to drive')
        r = c.require(k.sh('%s support/editing-browser.py safety \'{}\'' % PLAYWRIGHT_PYTHON,
                           cwd=str(REPO / 'tests'), timeout=1200, label='browser safety'),
                      'the browser')
        SAFETY = json.loads(r.text)
        if SAFETY.get('skipped'):
            c.skip(SAFETY['skipped'])
        c.expect_true(SAFETY.get('original') is not None, 'the field had a value to begin with')

    with k.check('an edit survives the tab being hidden and the page reloaded') as c:
        if not SAFETY or SAFETY.get('skipped'):
            c.skip('the browser half did not run')
        expected = (SAFETY['original'] or '') + ' hidden'
        c.expect_equal(SAFETY['afterHidden'], expected, 'in storage when the tab went away')
        c.expect_equal(SAFETY['afterReload'], expected, 'and still there after the reload')

    with k.check('an edit made offline is kept and drains when the network is back') as c:
        if not SAFETY or SAFETY.get('skipped'):
            c.skip('the browser half did not run')
        offline = SAFETY['offline']
        expected = (SAFETY['original'] or '') + ' offline'
        c.expect_equal(offline['value'], expected, 'read back while offline')
        c.expect_true(offline['outbox'] >= 1, '%d change(s) waiting' % offline['outbox'])
        c.expect_true(offline['drained'], 'the queue drained')
        c.expect_equal(offline['valueAfterDrain'], expected, 'and the value is unchanged')

    with k.check('a second editor reads the change out of the repository') as c:
        if not SAFETY or SAFETY.get('skipped'):
            c.skip('the browser half did not run')
        c.expect_equal(SAFETY['secondEditorSees'], (SAFETY['original'] or '') + ' offline',
                       'what the second browser hydrated from the agent')

    with k.check('the live field is back to the value the run found') as c:
        if not SAFETY or SAFETY.get('skipped'):
            c.skip('the browser half did not run')
        # The invariant on the live site: the browser's storage was emptied and
        # this value came from the agent.
        c.expect_equal(SAFETY['readBack'], SAFETY['original'],
                       'read back out of an emptied browser')

print(json.dumps(SAFETY, indent=1))

{
 "scenario": "safety",
 "original": "Our fleet",
 "afterHidden": "Our fleet hidden",
 "afterReload": "Our fleet hidden",
 "offline": {
  "value": "Our fleet offline",
  "outbox": 1,
  "saveState": "offline",
  "drained": true,
  "valueAfterDrain": "Our fleet offline"
 },
 "secondEditorSees": "Our fleet offline",
 "readBack": "Our fleet"
}


## The half second before the store, which is where an edit was lost

`editing-performance.md` holds the invariant with a synchronous write on
`visibilitychange` to hidden and on `pagehide`, and that write is of the store.
A field's change was not a store change until its own 500 ms debounce had
fired, and only a blur ever started that debounce, so a tab that went away
inside the window wrote a store that had never heard of the edit. Measured on
the deployed booklimo.at on 2026-09-08 and reported in `draft-state.md`, "The
one that failed": a close at 0 ms or 300 ms after the blur left the previous
value in `localStorage` with an empty outbox, and typing without leaving the
field was covered by nothing at all.

`packages/jaen/src/utils/on-leave.ts` makes the way out one ordered pass, the
field's debounce first and the store's write after it, and `TextField`
dispatches while a person types. This is the gate on both. Every reading is the
value in `localStorage` at the instant the tab is hidden, which is what a
browser that never comes back would have left behind.


In [16]:
LOSSES = None

with k.section('the half second before the store'):
    with k.check('the tab goes away with nothing waited out') as c:
        if not os.path.isfile(PLAYWRIGHT_PYTHON):
            c.skip('no playwright interpreter at %s' % PLAYWRIGHT_PYTHON)
        if not (SITE / 'public' / 'index.html').is_file():
            c.skip('no production build of booklimo.at in %s' % (SITE / 'public'))
        if not os.path.isfile(os.path.expanduser('~/.config/taxi-app/humans.env')):
            c.skip('no booklimo human admin configured')
        if not AGENT_IN_BUILD:
            c.skip('this build carries no agent option, so there is no shared draft to drive')
        r = c.require(k.sh('%s support/editing-browser.py losses \'{}\'' % PLAYWRIGHT_PYTHON,
                           cwd=str(REPO / 'tests'), timeout=1800, label='browser losses'),
                      'the browser')
        LOSSES = json.loads(r.text)
        if LOSSES.get('skipped'):
            c.skip(LOSSES['skipped'])
        c.expect_true(LOSSES.get('original') is not None, 'the field had a value to begin with')

    with k.check('the edit is in localStorage the moment the tab goes') as c:
        if not LOSSES or LOSSES.get('skipped'):
            c.skip('the browser half did not run')
        for hide in LOSSES['hides']:
            c.expect_equal(hide['inStorageWhenHidden'], hide['typed'],
                           '%d ms after the blur' % hide['delayMs'] if hide['blurred']
                           else 'with the caret still in the field')

    with k.check('and the object has it once the tab comes back') as c:
        if not LOSSES or LOSSES.get('skipped'):
            c.skip('the browser half did not run')
        for hide in LOSSES['hides']:
            c.expect_equal(hide['afterDrain'], hide['typed'],
                           'after the queue drained, %d ms case' % hide['delayMs'])

    with k.check('a real tab close with the caret in the field loses nothing') as c:
        if not LOSSES or LOSSES.get('skipped') or not LOSSES.get('close'):
            c.skip('the browser half did not run')
        close = LOSSES['close']
        c.expect_equal(close['inStorageAfterClose'], close['typed'],
                       'read out of a second tab of the same browser')
        if close.get('signedInAgain'):
            c.expect_equal(close.get('afterDrain'), close['typed'],
                           'and the object took it when the CMS was signed in again')

    with k.check('the live field is back to the value this scenario found') as c:
        if not LOSSES or LOSSES.get('skipped'):
            c.skip('the browser half did not run')
        c.expect_equal(LOSSES['readBack'], LOSSES['original'],
                       'read back out of an emptied browser')

print(json.dumps(LOSSES, indent=1))


{
 "scenario": "losses",
 "hides": [
  {
   "delayMs": 0,
   "blurred": true,
   "typed": "Our fleet l0",
   "inStorageWhenHidden": "Our fleet l0",
   "stateWhenHidden": {
    "outbox": 1,
    "saveState": "saving",
    "revision": 132
   },
   "lostAtHide": false,
   "afterDrain": "Our fleet l0"
  },
  {
   "delayMs": 300,
   "blurred": true,
   "typed": "Our fleet l300",
   "inStorageWhenHidden": "Our fleet l300",
   "stateWhenHidden": {
    "outbox": 1,
    "saveState": "saving",
    "revision": 133
   },
   "lostAtHide": false,
   "afterDrain": "Our fleet l300"
  },
  {
   "delayMs": 0,
   "blurred": false,
   "typed": "Our fleet l0n",
   "inStorageWhenHidden": "Our fleet l0n",
   "stateWhenHidden": {
    "outbox": 1,
    "saveState": "saving",
    "revision": 134
   },
   "lostAtHide": false,
   "afterDrain": "Our fleet l0n"
  }
 ],
 "close": {
  "typed": "Our fleet close",
  "inStorageAfterClose": "Our fleet close",
  "state": {
   "outbox": 1,
   "saveState": "saving",
   "revis

## What this run leaves for the reviewer

- The two red checks are the plan's changes 2 and 3 and nothing else. Every
  other check is green against the code as it stands, which is the point: the
  CMS does not lose an edit today, and the change must not make that untrue.
- The offline retry observation is a documentation defect, not a loss: the
  first retry is at five seconds rather than the documented two.
- The browser half was run against the live agent on booklimo. It commits, and
  it sets back. If the last check is ever red, the field on booklimo.at is left
  holding the value the run typed and a person has to put it right.
- What is not covered here: two editors writing the *same* field at the same
  moment, which is the agent's rebase and belongs to `draft-state.md`; and a
  browser killed between the dispatch and the write, which cannot be staged
  from inside the page and is argued rather than measured.

In [17]:
k.summary()
k.save_results('results-10-draft-persistence.json')
rc = k.verdict()
# Changes 2 and 3 of editing-performance.md have not been made yet, so this
# baseline run is expected to end here. The assertion stays: after the change
# every check in this notebook has to be green.
assert rc == 0, 'run has FAILures — see the summary above'

#,Status,Section,Check,Evidence
1,PASS,the harness,the harness bundles out of the jaen source,bundled 1761858 B
2,PASS,the fixture,the head the transition kept is beside the site,live.json at /home/snekmin/git/limosen-v3/booklimo.at-transition-before.head. live-media.json at /home/snekmin/git/limosen-v3/booklimo.at-transition-before.head. 80213 B of draft at /tmp/jaen-editing/booklimo-draft.json
3,PASS,the wire,every document the client sends validates against the agent's schema,"7 document(s): JaenAgentDraft, JaenAgentDiscardPreview, JaenAgentDiscard, JaenAgentSave, Jaen.... JaenAgentDraft. JaenAgentDiscardPreview. JaenAgentDiscard. JaenAgentSave. JaenAgentViewer. JaenAgentSubscribe. JaenAgentPublish. draft, save, viewer, subscribe and publish"
4,PASS,the persisted payload,the payload carries the unsent change and the edit,"the edit is in the payload, read back out of it. changes waiting in the outbox. what is waiting"
5,PASS,the persisted payload,the payload does not carry the media catalogue,"1086 B payload, 0 media nodes in it, 2 B of catalogue. change 2 of the plan: the catalogue is not persisted"
6,PASS,a reload,an edit survives the browser being started again,the value the second process read out of storage. and 0 media nodes came back with it
7,PASS,a hidden tab,a hidden tab has the edit in storage with no waiting,3 listener(s) heard visibilitychange. read back out of storage in the same frame as the event
8,WARN,a hidden tab,the edit is in storage within the idle deadline anyway,"read back 300 ms after the dispatch. the write is asynchronous: nothing was in storage in the dispatch's own frame, which is what ..."
9,PASS,offline,"an edit made offline is kept, in the store and in storage",read back out of storage while the agent is unreachable. waiting in the outbox. what the toolbar says
10,PASS,offline,the queue drains on its own when the agent comes back,the outbox is empty again. and the value is still the one that was typed. drained 2.0 s after the network came back
